# THE MF MODEL IS ALREADY PRETRAINED, USE IN CAUTIONS

In [ ]:
import torch
print(torch.__version__)
from time import time
from utils import *
from models import MF
import torch.optim as optim

# Change only these 5 variable & run the notebook, train MF on target domain
gpu_id = 0
source_domain = "book"
target_domain = "movie"
# ratio of excluded overlapping user's interaction
ratio = "20"
# True = Ignore ratio & train 100% of interaction, use this if training target domain
# False = Use the ratio, use this if training source domain, to avoid leakage of test overlapping user's interaction
noExcludeUser = True

In [ ]:
device = torch.device(f'cuda:{gpu_id}' if torch.cuda.is_available() and gpu_id < torch.cuda.device_count() else 'cpu')
amazon_json = f"amazon_{target_domain}"
pt_path = os.path.join(train_data[f"{source_domain}_{target_domain}_save"], f"{ratio}train.pt")
if (noExcludeUser):
    weight_save = mf_weight[f"{target_domain}_100"]
else:
    weight_save = mf_weight[f"{source_domain}_{target_domain}_{ratio}"]
user_map_path, item_map_path = user_map_paths[target_domain], item_map_paths[target_domain]

# exclude the overlapping test user from MF train
train_uid, train_iid, train_rates, tn_user, tn_item = \
    get_pretrain_data(noExcludeUser, pt_path, amazon_json, user_map_path, item_map_path, device)

# Mdel,
lightgcn_source = MF(tn_user, tn_item, EMBBEDDING_DIM)
lightgcn_source = lightgcn_source.to(device)
# Optimizer
optimizer = optim.Adam(
    list(lightgcn_source.parameters()),
    lr=MF_LR,
    # weight_decay=1e-5
)

train_losses=[]

s = time()
for iter in range(ITERATIONS):
    lightgcn_source.train()
    for batch_uid, batch_iid, batch_rates in mini_batch_iterator(train_uid, train_iid, train_rates, batch_size=BATCH_SIZE):
        user_indices, item_indices, true_ratings\
            = batch_uid.to(device), batch_iid.to(device), batch_rates.to(device)
        # Source domain: all users
        predicted_ratings = lightgcn_source(user_indices, item_indices) 
        # rating calculation
        true_ratings = (true_ratings - RATES_ADD) / RATES_MULTIPLY
        train_loss = MSELOSS(predicted_ratings, true_ratings)
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
    if iter % ITERS_PER_EVAL == 0:
        lightgcn_source.eval()
        with torch.no_grad():
            train_losses.append(train_loss.item())
        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)} \
              mae: {round(mae(predicted_ratings, true_ratings).item(), 5)} rmse: {round(rmse(predicted_ratings, true_ratings).item(), 5)}")
end = time()
print(f"Costing {end-s}s for training {ITERATIONS} iterations")

torch.save(lightgcn_source.state_dict(), weight_save)